# Capítulo 9 — Monte Carlo: calcular mediante azar

**Cuaderno interactivo de *La servilleta y el ordenador*.**

Cada sección reproduce una figura del capítulo. La gracia no es ejecutarlas: es **cambiar los parámetros y comprobar si ocurre lo que esperabas**.

> Antes de ejecutar cada celda, escribe en una línea qué esperas ver. Después mira si ocurrió. Y después, por qué.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / ''))
sys.path.insert(0, '../../../herramientas')
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / 'herramientas'))

import numpy as np
import matplotlib.pyplot as plt
from estilo_libro import C, use_style, rng, save

use_style()
%matplotlib inline

---

## La aguja de Buffon: ¿cómo puede tirar agujas darte pi?

Simulación del experimento y convergencia del estimador, con el resultado
«demasiado bueno» de Lazzarini (1901) marcado para comparar.

La figura responde: ¿cuántas agujas hacen falta para dos decimales de pi, y por
qué el resultado de Lazzarini es sospechoso?

Ejecutar:  python fig_buffon.py

*(script original: `codigo/fig_buffon.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(1777)

L, D = 1.0, 1.0          # longitud de la aguja y separación entre líneas
N = 2_000_000

# Centro uniforme respecto a la línea más cercana, ángulo uniforme
y = r.uniform(0, D / 2, N)
theta = r.uniform(0, np.pi / 2, N)
cruza = y <= (L / 2) * np.sin(theta)

n = np.arange(1, N + 1)
p_estimada = np.cumsum(cruza) / n
with np.errstate(divide="ignore"):
    pi_estimada = 2 * L / (D * p_estimada)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 4.2),
                               gridspec_kw={"width_ratios": [1, 1.35]})

# --- Panel 1: el experimento ---------------------------------------------
m = 220
for k in range(m):
    xc = r.uniform(0.2, 3.8)
    yc = r.uniform(0.2, 3.8)
    th = r.uniform(0, np.pi)
    dx, dy = (L / 2) * np.cos(th), (L / 2) * np.sin(th)
    corta = int(np.floor(yc - dy)) != int(np.floor(yc + dy))
    ax1.plot([xc - dx, xc + dx], [yc - dy, yc + dy],
             color=C.red if corta else C.grey, lw=1.0,
             alpha=0.9 if corta else 0.5)
for yl in range(5):
    ax1.axhline(yl, color=C.ink, lw=1.2)
ax1.set_xlim(0, 4), ax1.set_ylim(-0.1, 4.1)
ax1.set_aspect("equal"), ax1.axis("off")
ax1.set_title("220 agujas: en rojo las que cruzan", fontsize=10)

# --- Panel 2: convergencia -----------------------------------------------
paso = np.unique(np.logspace(0, np.log10(N), 400).astype(int)) - 1
ax2.semilogx(n[paso], pi_estimada[paso], color=C.blue, lw=1.2,
             label="estimación acumulada")
ax2.axhline(np.pi, color=C.ink, lw=1.4)
ax2.text(2, np.pi + 0.06, r"$\pi$", fontsize=11, color=C.ink)

# Banda teórica +-1 sigma
p = 2 * L / (np.pi * D)
sigma_pi = (2 * L / (D * p**2)) * np.sqrt(p * (1 - p) / n)
ax2.fill_between(n[paso], np.pi - sigma_pi[paso], np.pi + sigma_pi[paso],
                 color=C.blue, alpha=0.18, label=r"$\pm\sigma$ teórica")

ax2.plot(3408, 355 / 113, "*", color=C.red, ms=15, zorder=5)
ax2.annotate("Lazzarini (1901): 3408 agujas,\n"
             r"$\pi = 355/113$, seis decimales",
             xy=(3408, 355 / 113), xytext=(30, 3.55), fontsize=8.4, color=C.red,
             arrowprops=dict(arrowstyle="->", color=C.red, lw=1.0))
ax2.set_ylim(2.6, 3.75)
ax2.set_xlabel("número de agujas $N$")
ax2.set_ylabel(r"estimación de $\pi$")
ax2.set_title(r"Convergencia: el margen baja como $1/\sqrt{N}$")
ax2.legend(fontsize=8, loc="lower right")

print(f"con N={N:,}: pi ≈ {pi_estimada[-1]:.5f}")
print(f"sigma teórica en N=3408: {(2*L/(D*p**2))*np.sqrt(p*(1-p)/3408):.4f}")
print(f"error de Lazzarini: {abs(355/113 - np.pi):.2e}")
plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## 1/sqrt(N): ¿es una ley de la naturaleza o se puede batir?

Compara Monte Carlo puro, Monte Carlo cuasi-aleatorio (Sobol) e integración por
rejilla, en dimensión creciente.

La figura responde: ¿cuándo gana Monte Carlo a una rejilla, y por qué?

Ejecutar:  python fig_convergencia_mc.py

*(script original: `codigo/fig_convergencia_mc.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import qmc

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(9)


def integrando(x):
    """Función suave en [0,1]^d con integral exacta conocida."""
    return np.prod(np.cos(x * np.pi / 2) * (np.pi / 2), axis=-1)   # integral = 1


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 4.2))

# --- Panel 1: MC vs quasi-MC en d = 4 ------------------------------------
d = 4
Ns = np.unique(np.logspace(1.5, 5.5, 25).astype(int))
err_mc, err_qmc = [], []
for N in Ns:
    x = r.random((N, d))
    err_mc.append(abs(integrando(x).mean() - 1.0))
    s = qmc.Sobol(d=d, scramble=True, seed=3).random(N)
    err_qmc.append(abs(integrando(s).mean() - 1.0))

ax1.loglog(Ns, err_mc, "o-", color=C.blue, ms=4, lw=1.3,
           label="Monte Carlo puro")
ax1.loglog(Ns, err_qmc, "s-", color=C.green, ms=4, lw=1.3,
           label="cuasi-Monte Carlo (Sobol)")
ax1.loglog(Ns, 0.35 / np.sqrt(Ns), ":", color=C.ink, lw=1.4,
           label=r"$N^{-1/2}$")
ax1.loglog(Ns, 3.0 / Ns, ":", color=C.grey, lw=1.4, label=r"$N^{-1}$")
ax1.set_xlabel("número de muestras $N$")
ax1.set_ylabel("error absoluto")
ax1.set_title("Dimensión $d=" + str(d) + r"$: hay algo mejor que $1/\sqrt{N}$")
ax1.legend(fontsize=8, loc="lower left")

# --- Panel 2: la maldición (y la bendición) de la dimensión --------------
dims = np.arange(1, 13)
N_OBJETIVO = 10_000
err_mc_d, err_rejilla_d = [], []
for dd in dims:
    x = r.random((N_OBJETIVO, dd))
    err_mc_d.append(abs(integrando(x).mean() - 1.0))
    # rejilla con el mismo presupuesto de puntos: n por eje
    n_eje = max(int(round(N_OBJETIVO ** (1 / dd))), 2)
    ejes = (np.arange(n_eje) + 0.5) / n_eje
    malla = np.stack(np.meshgrid(*([ejes] * dd), indexing="ij"), axis=-1)
    err_rejilla_d.append(abs(integrando(malla.reshape(-1, dd)).mean() - 1.0))

ax2.semilogy(dims, err_mc_d, "o-", color=C.blue, ms=5, lw=1.6,
             label="Monte Carlo ($10^4$ puntos)")
ax2.semilogy(dims, err_rejilla_d, "s-", color=C.red, ms=5, lw=1.6,
             label="rejilla ($10^4$ puntos)")
ax2.set_xlabel("dimensión $d$")
ax2.set_ylabel("error absoluto")
ax2.set_title("El mismo presupuesto, dos estrategias")
ax2.set_ylim(1e-10, 3e1)
ax2.legend(fontsize=8, loc="lower right")
ax2.annotate("a partir de aquí\nla rejilla es inservible",
             xy=(6, err_rejilla_d[5]), xytext=(1.4, 4e0), fontsize=8.4,
             color=C.red, arrowprops=dict(arrowstyle="->", color=C.red, lw=1.0))
ax2.text(2.4, 8e-9, "el error de Monte Carlo\nNO depende de $d$",
         fontsize=8.6, color=C.blue)

print("d, err_MC, err_rejilla")
for dd, a, b in zip(dims, err_mc_d, err_rejilla_d):
    print(f"{dd:2d}  {a:.2e}  {b:.2e}")
plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## Metropolis: ¿cómo se muestrea algo que no sabes normalizar?

Cadena de Metropolis sobre una distribución bimodal, con tres tamaños de
paso. Se muestran la traza, el histograma y la autocorrelación.

La figura responde: ¿cómo se ve una cadena que parece convergida y no lo está?

Ejecutar:  python fig_metropolis.py

*(script original: `codigo/fig_metropolis.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(1953)


def log_p(x):
    """Bimodal: dos gaussianas separadas. No está normalizada, y da igual."""
    return np.logaddexp(-0.5 * ((x + 3) / 0.6) ** 2,
                        -0.5 * ((x - 3) / 0.6) ** 2)


def metropolis(paso, n=60_000, x0=-3.0):
    """Metropolis con paso gaussiano. Los sorteos se generan de golpe."""
    saltos = r.normal(0.0, paso, n)
    log_u = np.log(r.random(n))
    x, lp = x0, log_p(x0)
    cadena = np.empty(n)
    aceptados = 0
    for i in range(n):
        propuesta = x + saltos[i]
        lp_nuevo = log_p(propuesta)
        if log_u[i] < lp_nuevo - lp:
            x, lp = propuesta, lp_nuevo
            aceptados += 1
        cadena[i] = x
    return cadena, aceptados / n


def autocorr(x, maxlag=400):
    """Autocorrelación por FFT: O(N log N) en lugar de O(N^2)."""
    x = x - x.mean()
    n = 1 << (2 * len(x) - 1).bit_length()
    f = np.fft.rfft(x, n)
    c = np.fft.irfft(f * np.conj(f), n)[:maxlag]
    return c / c[0]


PASOS = [(0.2, "paso 0,2 — apenas se mueve"),
         (1.5, "paso 1,5 — aceptación «de manual»"),
         (8.0, "paso 8,0 — el único que cruza")]
fig, axes = plt.subplots(3, 3, figsize=(11.4, 6.6),
                         gridspec_kw={"width_ratios": [1.5, 1, 1],
                                      "hspace": 0.55, "wspace": 0.3})

xx = np.linspace(-6, 6, 400)
densidad = np.exp(log_p(xx))
densidad /= np.trapezoid(densidad, xx) if hasattr(np, "trapezoid") else np.trapz(densidad, xx)

for fila, (paso, titulo) in enumerate(PASOS):
    cadena, tasa = metropolis(paso)
    color = [C.red, C.green, C.ochre][fila]

    ax = axes[fila, 0]
    ax.plot(cadena[:8000], color=color, lw=0.5)
    ax.set_ylabel("$x$")
    ax.set_title(f"{titulo} — aceptación {tasa:.0%}", fontsize=9.5)
    ax.set_ylim(-6, 6)
    if fila == 2:
        ax.set_xlabel("iteración")

    ax = axes[fila, 1]
    ax.hist(cadena[5000:], bins=80, density=True, color=color, alpha=0.55,
            edgecolor="none")
    ax.plot(xx, densidad, color=C.ink, lw=1.6)
    ax.set_yticks([])
    ax.set_title("histograma frente a la verdad", fontsize=9)
    if fila == 2:
        ax.set_xlabel("$x$")

    ax = axes[fila, 2]
    ac = autocorr(cadena[5000:])
    ax.plot(ac, color=color, lw=1.4)
    ax.axhline(0, color=C.ink, lw=0.9)
    tau = 1 + 2 * np.sum(ac[:200])
    ax.set_title(f"autocorrelación, $\\tau_{{int}}\\approx${tau:.0f}",
                 fontsize=9)
    ax.set_ylim(-0.2, 1.05)
    if fila == 2:
        ax.set_xlabel("retardo")
    print(f"{titulo:24s} aceptación {tasa:5.1%}  tau_int ≈ {tau:6.0f}  "
          f"N_eficaz ≈ {len(cadena[5000:]) / max(tau, 1):.0f}")

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Cómo se generan muestras de una distribución que no viene en la biblioteca?

Tres técnicas: transformada inversa, rechazo y muestreo por importancia, con
la reducción de varianza que consigue la tercera.

La figura responde: ¿por qué muestrear «donde importa» reduce el error sin
introducir sesgo?

Ejecutar:  python fig_muestreo.py

*(script original: `codigo/fig_muestreo.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(13)

fig, axes = plt.subplots(1, 3, figsize=(11.6, 3.9))

# --- 1. Transformada inversa ---------------------------------------------
ax = axes[0]
u = np.linspace(0.001, 0.999, 400)
x = -np.log(1 - u)                       # inversa de la exponencial
ax.plot(u, x, color=C.blue, lw=2)
for uu in [0.2, 0.5, 0.8, 0.95]:
    xx = -np.log(1 - uu)
    ax.plot([0, uu, uu], [xx, xx, 0], color=C.red, lw=1.0, alpha=0.8)
    ax.plot(uu, 0, "o", color=C.red, ms=4)
ax.set_xlabel("$u$ uniforme en [0,1]"), ax.set_ylabel("$x = F^{-1}(u)$")
ax.set_title("Transformada inversa\n$x=-\\ln(1-u)$", fontsize=9.5)
ax.set_xlim(0, 1), ax.set_ylim(0, 3.5)

# --- 2. Rechazo -----------------------------------------------------------
ax = axes[1]
def objetivo(x):
    return np.exp(-x**2 / 2) * (1 + 0.7 * np.sin(4 * x)**2)

xx = np.linspace(-3.5, 3.5, 400)
M = 1.75
ax.plot(xx, objetivo(xx), color=C.blue, lw=2, label="$p(x)$ (sin normalizar)")
ax.plot(xx, M * np.exp(-xx**2 / 2), "--", color=C.ochre, lw=1.6,
        label="$M q(x)$ propuesta")
n = 500
xs = r.normal(0, 1, n)
us = r.uniform(0, M * np.exp(-xs**2 / 2), n)
acepta = us <= objetivo(xs)
ax.plot(xs[acepta], us[acepta], ".", color=C.green, ms=3, alpha=0.75)
ax.plot(xs[~acepta], us[~acepta], ".", color=C.red, ms=3, alpha=0.5)
ax.set_xlabel("$x$"), ax.set_ylabel("altura sorteada")
ax.set_title(f"Rechazo\naceptación = {acepta.mean():.0%}", fontsize=9.5)
ax.legend(fontsize=7.4, loc="upper right")

# --- 3. Importancia -------------------------------------------------------
ax = axes[2]
# Estimar P(X > 4) para una normal estándar: suceso raro
UMBRAL = 4.0
exacto = 3.167124e-5
Ns = np.unique(np.logspace(2, 6, 20).astype(int))
err_directo, err_import = [], []
for N in Ns:
    directo = (r.normal(0, 1, N) > UMBRAL).mean()
    err_directo.append(abs(directo - exacto) / exacto)
    # propuesta desplazada al umbral
    y = r.normal(UMBRAL, 1, N)
    peso = np.exp(-y**2 / 2) / np.exp(-(y - UMBRAL)**2 / 2)
    estimador = (peso * (y > UMBRAL)).mean()
    err_import.append(abs(estimador - exacto) / exacto)

ax.loglog(Ns, np.maximum(err_directo, 1e-6), "o-", color=C.red, ms=4, lw=1.4,
          label="muestreo directo")
ax.loglog(Ns, err_import, "s-", color=C.green, ms=4, lw=1.4,
          label="por importancia")
ax.set_xlabel("número de muestras $N$")
ax.set_ylabel("error relativo")
ax.set_title(r"Suceso raro: $P(X>4)=3{,}2\times10^{-5}$", fontsize=9.5)
ax.legend(fontsize=8)
ax.annotate("con $10^3$ muestras el directo\nno ve ni un suceso",
            xy=(1e3, 1.0), xytext=(3e3, 0.12), fontsize=8, color=C.red,
            arrowprops=dict(arrowstyle="->", color=C.red, lw=1.0))

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 
